# V32 — V4-pure-prune (X35 Q3+Q4 subtraction)

Submission = V4 banked 154-flag set minus 23 rows whose X35 sits in train quintiles Q3 or Q4.

Train X35 quintile defect rates: Q0=11.8%, Q1=9.6%, Q2=1.1%, Q3=0.7%, Q4=1.1%.  
88% of train defects live in Q0+Q1. V4's OOF precision in Q3+Q4 is ~3% vs ~12% in Q0+Q1, suggesting overflagging there.  
Pure subtraction → Spearman with V4_binary = 1.0 on kept rows.

In [ ]:
import pandas as pd
test = pd.read_csv('test.csv')
train = pd.read_csv('train.csv')
v4_pred = pd.read_csv('v4_expected_submission.csv')  # V4 banked 154-flag

# X35 quintile cutoffs from train
qcuts = pd.qcut(train['X35'], 5, retbins=True, duplicates='drop')[1]
test['X35_q'] = pd.cut(test['X35'], bins=qcuts, labels=['Q0','Q1','Q2','Q3','Q4'], include_lowest=True)

# Identify V4-flagged in Q3+Q4 (low-defect-rate buckets per train)
v4_flagged = set(v4_pred[v4_pred.Y == 1].CoilID)
test['v4_flag'] = test.CoilID.isin(v4_flagged).astype(int)
prune = test[(test.v4_flag == 1) & (test.X35_q.isin(['Q3','Q4']))].CoilID.tolist()

# V32 = V4 minus 23 Q3+Q4 picks
v32 = v4_pred.copy()
v32.loc[v32.CoilID.isin(prune), 'Y'] = 0
v32.to_csv('solution.csv', index=False)
print(f'V32 flagged: {(v32.Y == 1).sum()} of {len(v32)}')